In [1]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from LangChain.app.config import config
from IPython.display import Image

# 初始化 LLM
llm = ChatOpenAI(
    model=config.CONFIG_DEEPSEEK["model"],
    openai_api_key=config.CONFIG_DEEPSEEK["api_key"],
    openai_api_base=config.CONFIG_DEEPSEEK["base_url"],
    temperature=0.7
)


# 路由函数
def classify_intent(state: MessagesState) -> str:
    """根据用户意图路由到不同的 Agent"""
    last_message = state["messages"][-1]
    content = last_message.content.lower()

    if "天气" in content or "温度" in content:
        return "weather_agent"
    elif "代码" in content or "编程" in content:
        return "code_agent"
    elif "再见" in content or "退出" in content:
        return "farewell"
    else:
        return "general_agent"


# 定义各个 Agent 节点
def router_node(state: MessagesState) -> dict:
    """路由节点：不做处理，只用于触发路由判断"""
    return {}


def weather_node(state: MessagesState) -> dict:
    """天气 Agent"""
    response = llm.invoke([
        SystemMessage(content="你是一个天气助手，友好地回答天气相关问题。如果没有实时数据，可以给出一般性建议。"),
        *state["messages"]
    ])
    return {"messages": [response]}


def code_node(state: MessagesState) -> dict:
    """代码 Agent"""
    response = llm.invoke([
        SystemMessage(content="你是一个编程助手，擅长解答代码问题并给出清晰的代码示例。"),
        *state["messages"]
    ])
    return {"messages": [response]}


def general_node(state: MessagesState) -> dict:
    """通用 Agent"""
    response = llm.invoke([
        SystemMessage(content="你是一个友善的 AI 助手，可以回答各种问题。"),
        *state["messages"]
    ])
    return {"messages": [response]}


def farewell_node(state: MessagesState) -> dict:
    """告别节点"""
    return {"messages": [{"role": "assistant", "content": "再见！期待下次与你交流。"}]}


# 构建图
builder = StateGraph(MessagesState)

# 添加节点
builder.add_node("router", router_node)
builder.add_node("weather_agent", weather_node)
builder.add_node("code_agent", code_node)
builder.add_node("general_agent", general_node)
builder.add_node("farewell", farewell_node)

# 添加边
builder.add_edge(START, "router")
builder.add_conditional_edges(
    "router",
    classify_intent,
    {
        "weather_agent": "weather_agent",
        "code_agent": "code_agent",
        "general_agent": "general_agent",
        "farewell": "farewell",
    }
)

# 所有 agent 节点处理完后结束
for node in ["weather_agent", "code_agent", "general_agent", "farewell"]:
    builder.add_edge(node, END)

# 编译图
graph = builder.compile()

Image(graph.get_graph().draw_mermaid_png())

# 或者打印 Mermaid 格式
print(graph.get_graph().draw_mermaid())

D:\anaconda3\envs\langchain-py311\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	router(router)
	weather_agent(weather_agent)
	code_agent(code_agent)
	general_agent(general_agent)
	farewell(farewell)
	__end__([<p>__end__</p>]):::last
	__start__ --> router;
	router -.-> code_agent;
	router -.-> farewell;
	router -.-> general_agent;
	router -.-> weather_agent;
	code_agent --> __end__;
	farewell --> __end__;
	general_agent --> __end__;
	weather_agent --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [2]:
# 测试不同意图
test_inputs = [
    "北京今天天气怎么样？",
    "帮我写一个 Python 快速排序",
    "你好，介绍一下你自己",
    "再见啦！"
]

for user_input in test_inputs:
    print(f"\n用户: {user_input}")
    result = graph.invoke({"messages": [HumanMessage(content=user_input)]})
    print(f"助手: {result['messages'][-1].content[:100]}...")
    print("-" * 50)
    



用户: 北京今天天气怎么样？
助手: 抱歉，我无法获取实时天气数据。不过我可以给你一些一般性建议：北京今天的天气通常可以参考当地天气预报或使用天气App。如果你能告诉我具体日期，我可以给你一些季节性的着装建议，比如冬天记得带羽绒服，夏天注...
--------------------------------------------------

用户: 帮我写一个 Python 快速排序
助手: 我来为你写一个快速排序的 Python 实现，包含基础版和优化版。

## 基础版快速排序

```python
def quick_sort(arr):
    """
    快速排序基础版本
 ...
--------------------------------------------------

用户: 你好，介绍一下你自己
助手: 你好！很高兴认识你！我是 DeepSeek，一个由深度求索公司开发的 AI 助手。我的主要任务就是帮助大家解答问题、提供信息、协助完成任务，或者只是聊聊天、陪你度过一段愉快的时光。

我可以做的事情包...
--------------------------------------------------

用户: 再见啦！
助手: 再见！期待下次与你交流。...
--------------------------------------------------
